# Experiment 8: **Behavioural Dynamics**

## Objective
In this experiment, we measure the development of the negotiations for different behavioural configurations on the base game.

---

## Methodology

- **Models Used in Our Experiment:**  
  - GPT-4o Mini  
  - Qwen2.5-72B  

- **Evaluation Metrics:**  
  The script computes the following metrics for each behavioural configuration:  
  - % 5/6-way agreement
  - % 6-way agreement 
  - % Any agreement

- **Behavioural Configurations:**  
  - All cooperative (default mode)  
  - One greedy (A player in concordance with player 1, i.e., the Environmental League)  
  - One greedy (Player 1)  
  - Two greedy (Two players in discordance with player 1, i.e., the Mayor and the Local Labour Union)  
  - All greedy  
  - Adversarial untargeted (The adversarial player is the Environmental League)  
  - Adversarial targeted (The adversarial player is the Environmental League and the Local Labour Union is the target)  

---

## Results
The results of this experiment analyze how different behavioural dynamics affect negotiations in the base game. These findings are presented in **Table 10** and **Figure 4** of our paper.


In [6]:
import os
import eval_utils as evaluation
import json
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt


raw_path = '../our_games_descriptions/base/output/changing_behaviour'

variations = [
    "base",
    "one_greedy",
    "one_greedy_p1",
    "two_greedy",
    "all_greedy",
    "adversarial_untargeted",
    "adversarial_targeted",
]

models = [
    "gpt4o-mini",
    "Qwen2.5-72B-Instruct"
]

results = {}
ISSUES_NUM = 5
AGENTS_NUM = 6

for model in models:
    for variation in variations:
            
        # Set the directory
        if variation == 'base':
            directory = '../our_games_descriptions/base/output/original_code/{model}'.format(model=model)
        else:
            directory = os.path.join(raw_path, model, variation)

        # Load the data
        agents, role_to_agents, incentive_to_agents = evaluation.load_setup(directory, AGENTS_NUM, num_issues=ISSUES_NUM)
        answers_files = [ os.path.join(directory,filename) for filename in os.listdir(directory) if filename.startswith("history")]

        num_rounds = 0
        for file_ in answers_files:
            answers = json.load(open(file_))
            _num_rounds = len(answers['rounds'])
            num_rounds = max(num_rounds, _num_rounds)

        # Track statistics
        feasible_in_last_step = 0
        accepted_by_all_in_last_step = 0
        contained_feasible_deal = 0
        successfull_games = 0
        total_rounds = 0

        # Loop through all answer files (each represents a game)
        for file_ in answers_files:
            answers = json.load(open(file_))
            
            if len(answers['rounds']) != num_rounds:
                print(f"WARNING: Game {file_} has a different number of rounds")
                continue
            total_rounds += len(answers['rounds'])
            successfull_games += 1

            # Extract deals for this game
            feasible_found = False

            # Extract the name of the first player (p1) to validate feasibility throughout the game
            p1_name = answers['rounds'][0]['agent']

            total_deals = 0
            
            for i, round_ in enumerate(answers['rounds']):
                name, answer = round_['agent'], round_['public_answer']
                deal_unformatted, issues_suggested = evaluation.extract_deal(answer, ISSUES_NUM)

                try:
                    deal = evaluation.format_deal(deal_unformatted, ISSUES_NUM)
                except:
                    print(f"Error in game {file_} round {i}")
                    continue

                if issues_suggested >= ISSUES_NUM:
                    total_deals += 1

                # Check if the deal was feasible at any point (Deal must have been proposed by p1)
                if evaluation.is_feasible(agents, deal) and name == p1_name:
                    feasible_found = True
            

            # CHECK GAME COMPLETION METRICS

            last_deal = evaluation.format_deal(evaluation.extract_deal(answers['rounds'][-1]['public_answer'], ISSUES_NUM)[0], ISSUES_NUM)
            
            # 1. Check if the last deal is feasible
            if evaluation.is_feasible(agents, last_deal):
                feasible_in_last_step += 1

            # 2. Check if the last deal is acceptable by all agents
            all_accept = all(evaluation.calculator(agents[agent]["scores"], last_deal, ISSUES_NUM, verbose=False) >= agents[agent]["scores"]["min"] for agent in agents)
            if all_accept:
                accepted_by_all_in_last_step += 1

            # 3. Check if any deal during the game was in the feasibility set
            if feasible_found:
                contained_feasible_deal += 1

        # Compute percentages
        num_games = successfull_games
        perc_feasible_last = round((feasible_in_last_step / num_games) * 100, 2)
        perc_accepted_all_last = round((accepted_by_all_in_last_step / num_games) * 100, 2)
        perc_feasible_any = round((contained_feasible_deal / num_games) * 100, 2)

        results[(model, variation)] = {
            "5/6-way (%)": perc_feasible_last,
            "6-way (%)": perc_accepted_all_last,
            "Any (%)": perc_feasible_any,
        }


# Create a dataframe
df = pd.DataFrame(results).T

# Display the dataframe
display(df)

5/6-way (%)  6-way (%)  Any (%)
gpt4o-mini           base                          55.00       5.00    90.00
                     one_greedy                    80.00      10.00    95.00
                     one_greedy_p1                 45.00       5.00    90.00
                     two_greedy                    95.00       5.00   100.00
                     all_greedy                    50.00       0.00    80.00
                     adversarial_untargeted        90.00       0.00   100.00
                     adversarial_targeted          68.42       5.26    94.74
Qwen2.5-72B-Instruct base                          85.00       0.00    95.00
                     one_greedy                    95.00       0.00   100.00
                     one_greedy_p1                 50.00       5.00    55.00
                     two_greedy                    75.00       0.00    85.00
                     all_greedy                    15.00       5.00    40.00
                     adversarial_untargeted        85.00       0.00   100.00
                     adversarial_targeted          85.00       0.00    95.00